---
## 1. Setup & Dependencies

In [ ]:
# Install required packages
!pip install -q kaggle gensim xgboost lightgbm wordcloud plotly
!pip install -q contractions

In [ ]:
import os
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import contractions

# sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, roc_curve, accuracy_score
)
from sklearn.preprocessing import LabelEncoder

# XGBoost / LightGBM
from xgboost import XGBClassifier
import lightgbm as lgb

# Gensim Word2Vec
from gensim.models import Word2Vec

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, LSTM, Bidirectional, Embedding,
    GlobalAveragePooling1D, BatchNormalization, Input
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TF version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU'))} device(s)")

# Download NLTK resources
for resource in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger', 'omw-1.4', 'punkt_tab']:
    nltk.download(resource, quiet=True)

---
## 2. Load the Dataset

In [ ]:
from google.colab import files
print("Upload your kaggle.json API key:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews --unzip

df = pd.read_csv('IMDB Dataset.csv')

print(f"Shape: {df.shape}")
print(f"\nClass distribution:")
print(df['sentiment'].value_counts())
df.head(3)

---
## 3. Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words('english'))

# Keep negation words — they carry strong sentiment signal
KEEP_WORDS = {
    'not', 'no', 'nor', 'neither', 'never', 'nothing', 'nowhere',
    'nobody', 'none', "n't", 'cannot', "isn't", "wasn't", "aren't",
    "weren't", "don't", "doesn't", "didn't", "won't", "wouldn't",
    "shouldn't", "couldn't", "hadn't", "haven't", "hasn't"
}
STOP_WORDS -= KEEP_WORDS


def clean_text(text: str, lemmatize: bool = True) -> str:
    """Full preprocessing pipeline for a single review."""
    # 1. Lowercase
    text = text.lower()
    # 2. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # 3. Expand contractions  (don't → do not)
    text = contractions.fix(text)
    # 4. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # 5. Remove special characters / punctuation (keep apostrophes for n't)
    text = re.sub(r"[^a-z\s']", ' ', text)
    # 6. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # 7. Tokenise
    tokens = word_tokenize(text)
    # 8. Remove stopwords (but keep negations)
    tokens = [t for t in tokens if t not in STOP_WORDS or t in KEEP_WORDS]
    # 9. Lemmatise
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    # 10. Drop very short tokens
    tokens = [t for t in tokens if len(t) > 1]
    return ' '.join(tokens)


print("Preprocessing all reviews (this may take 2–3 min)...")
t0 = time.time()
df['clean_review'] = df['review'].apply(clean_text)
print(f"Done in {time.time()-t0:.1f}s")

# Encode labels
df['label'] = (df['sentiment'] == 'positive').astype(int)

print("\nBefore vs After:")
idx = 0
print("ORIGINAL:", df['review'].iloc[idx][:200])
print("\nCLEAN   :", df['clean_review'].iloc[idx][:200])

In [ ]:
# Quick sanity check
print(f"Null clean reviews: {df['clean_review'].isna().sum()}")
print(f"Empty clean reviews: {(df['clean_review'].str.strip() == '').sum()}")
df = df[df['clean_review'].str.strip() != ''].reset_index(drop=True)
print(f"Final dataset size: {len(df)}")